# Quora Duplicate Question Detection
Complete notebook using Word2Vec + Random Forest.
Run the cells from top to bottom. Place `train.csv` in the same directory.

In [1]:
!pip install gensim beautifulsoup4 contractions nltk tqdm joblib scikit-learn


In [2]:
import os
import re
import string
import joblib
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
import contractions

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/krishnapathak/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/krishnapathak/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/krishnapathak/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
df = pd.read_csv("train.csv")
df = df.dropna().reset_index(drop=True)
print(df.shape)
df.head()


(404287, 6)


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = BeautifulSoup(text,"html.parser").get_text()
    text = contractions.fix(text)
    text = text.translate(str.maketrans("","",string.punctuation))
    text = re.sub(r"\d+"," ",text)
    text = re.sub(r"\s+"," ",text).strip()
    words=[]
    for w in text.split():
        if w not in stop_words:
            words.append(lemmatizer.lemmatize(w))
    return " ".join(words)

tqdm.pandas()
df["question1"]=df["question1"].progress_apply(clean_text)
df["question2"]=df["question2"].progress_apply(clean_text)


  0%|          | 0/404287 [00:00<?, ?it/s]

  0%|          | 0/404287 [00:00<?, ?it/s]

In [5]:
def create_features(row):
    q1=row["question1"]
    q2=row["question2"]
    s1=set(q1.split())
    s2=set(q2.split())
    common=len(s1&s2)
    total=len(s1)+len(s2)
    return pd.Series({
        "q1_len":len(q1),
        "q2_len":len(q2),
        "q1_words":len(q1.split()),
        "q2_words":len(q2.split()),
        "word_common":common,
        "word_total":total,
        "word_share": common/total if total else 0,
        "len_diff": abs(len(q1.split())-len(q2.split()))
    })

features=df.progress_apply(create_features,axis=1)


  0%|          | 0/404287 [00:00<?, ?it/s]

In [6]:
sentences=[q.split() for q in df.question1]+[q.split() for q in df.question2]

w2v=Word2Vec(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=2,
    workers=os.cpu_count(),
    epochs=20,
    sg=1
)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [7]:
def sentence_vector(sentence):
    vec=np.zeros(300,dtype=np.float32)
    cnt=0
    for w in sentence.split():
        if w in w2v.wv:
            vec+=w2v.wv[w]
            cnt+=1
    if cnt:
        vec/=cnt
    return vec

q1=np.vstack(df.question1.progress_apply(sentence_vector))
q2=np.vstack(df.question2.progress_apply(sentence_vector))


  0%|          | 0/404287 [00:00<?, ?it/s]

  0%|          | 0/404287 [00:00<?, ?it/s]

In [8]:
X=np.hstack([q1,q2,features.values])
y=df["is_duplicate"].values

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

rf=RandomForestClassifier(
    n_estimators=300,
    max_depth=35,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train,y_train)

pred=rf.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))


Accuracy: 0.8175319696257637
              precision    recall  f1-score   support

           0       0.81      0.94      0.87     51005
           1       0.85      0.62      0.71     29853

    accuracy                           0.82     80858
   macro avg       0.83      0.78      0.79     80858
weighted avg       0.82      0.82      0.81     80858



In [9]:
joblib.dump(rf,"rf.pkl")
w2v.save("word2vec.model")
print("Models saved.")


Models saved.


In [ ]:
ef predict_duplicate(q1,q2):
    q1=clean_text(q1)
    q2=clean_text(q2)

    s1=set(q1.split())
    s2=set(q2.split())
    common=len(s1&s2)
    total=len(s1)+len(s2)

    extra=np.array([
        len(q1),
        len(q2),
        len(q1.split()),
        len(q2.split()),
        common,
        total,
        common/total if total else 0,
        abs(len(q1.split())-len(q2.split()))
    ],dtype=np.float32)

    x=np.hstack([
        sentence_vector(q1),
        sentence_vector(q2),
        extra
    ]).reshape(1,-1)

    pred=rf.predict(x)[0]
    prob=rf.predict_proba(x).max()

    if pred:
        print(f"Duplicate ({prob*100:.2f}%)")
    else:
        print(f"Not Duplicate ({prob*100:.2f}%)")


In [14]:
predict_duplicate(
    "how to pursue my dreams",
    "how to dream at night"
)


Not Duplicate (66.07%)
